In [1]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import math

# ========================== 1. Prepare Population Data (2000–2032) ==========================
# Use NISR validated data + predictions:
# - 2000–2001: Back-calculated (exponential model, r=5.05%)
# - 2002–2024: NISR census/NISR
# - 2025–2032: NISR provided projections

population_data = {
    "Year": list(range(2000, 2033)),  # 2000 to 2032
    "Population": [
        289750,  # 2000 (back-calculated)
        304400,  # 2001 (back-calculated)
        320516,  # 2002 (census)
        337100,  # 2003 (interpolated)
        354300,  # 2004 (interpolated)
        372100,  # 2005 (interpolated)
        390500,  # 2006 (interpolated)
        409600,  # 2007 (interpolated)
        429400,  # 2008 (interpolated)
        450000,  # 2009 (interpolated)
        480774,  # 2010 (interpolated)
        502200,  # 2011 (interpolated)
        529561,  # 2012 (census)
        558200,  # 2013 (interpolated)
        588300,  # 2014 (interpolated)
        620000,  # 2015 (interpolated)
        653300,  # 2016 (interpolated)
        688300,  # 2017 (interpolated)
        725100,  # 2018 (interpolated)
        763800,  # 2019 (interpolated)
        804500,  # 2020 (interpolated)
        847300,  # 2021 (interpolated)
        879505,  # 2022 (census)
        919472,  # 2023 (estimate)
        967512,  # 2024 (estimate)
        1017220, # 2025 (NISR projection)
        1068544, # 2026 (NISR projection)
        1121662, # 2027 (NISR projection)
        1176771, # 2028 (NISR projection)
        1233799, # 2029 (NISR projection)
        1292489, # 2030 (NISR projection)
        1352812, # 2031 (NISR projection)
        1414990  # 2032 (NISR projection)
    ]
}

# Convert to DataFrame for easy handling
df = pd.DataFrame(population_data)

# Add a column to distinguish "Observed" (2000–2024) vs "Projected" (2025–2032)
df["Data_Type"] = df["Year"].apply(lambda x: "Observed (Census by NISR/Estimates)" if x <= 2024 else "Projected (NISR)")

# ========================== 2. Calculate Growth Trend (for Annotation) ==========================
# Fit a linear trendline to show growth rate (2000–2032)
x = df["Year"]
y = df["Population"]
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

# Annual growth rate (in people per year)
annual_growth = round(slope)
# R² value (goodness of fit)
r_squared = round(r_value**2, 3)

# ========================== 3. Create Publication-Ready Graph ==========================
# Set plot style (professional, clean)
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(12, 8))

# Plot observed vs projected data (different colors/markers)
for data_type, group in df.groupby("Data_Type"):
    if data_type == "Observed (Census by NISR/Estimates)":
        ax.plot(
            group["Year"], group["Population"], 
            color="#1f77b4", marker="o", markersize=6, linewidth=2.5,
            label=data_type, markerfacecolor="#ff7f0e"  # Blue line + orange markers
        )
    else:
        ax.plot(
            group["Year"], group["Population"], 
            color="#2ca02c", marker="s", markersize=5, linewidth=2.5, linestyle="--",
            label=data_type, markerfacecolor="#2ca02c"  # Green dashed line + green markers
        )

# Add trendline (2000–2032)
trendline = slope * x + intercept
ax.plot(x, trendline, color="#d62728", linewidth=2, linestyle=":", label=f"Trendline (R² = {r_squared})")

# ========================== 4. Add Annotations (Key Census Years) ==========================
# Highlight 2002, 2012, 2022 (census years) with labels
census_years = [2002, 2012, 2022]
for year in census_years:
    pop = df[df["Year"] == year]["Population"].values[0]
    ax.annotate(
        f"{year}\n{pop:,} people",
        xy=(year, pop),
        xytext=(year - 1, pop + 50000),  # Offset text to avoid overlap
        ha="center", fontsize=10, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7),
        arrowprops=dict(arrowstyle="->", color="black", lw=1.5)
    )

# ========================== 5. Customize Axes/Labels/Titles ==========================
# Axes labels (clear, with units)
ax.set_xlabel("Year", fontsize=14, fontweight="bold", labelpad=10)
ax.set_ylabel("Gasabo District Population", fontsize=14, fontweight="bold", labelpad=10)

# Title (descriptive, linked to LULC context)
ax.set_title(
    "Gasabo District Population Trends (2000–2032)",
    fontsize=16, fontweight="bold", pad=20
)

# Customize ticks (show every 2 years for readability)
ax.set_xticks(range(2000, 2033, 2))
ax.tick_params(axis="both", labelsize=12)

# Format y-axis to show population in "thousands" (easier to read)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x/1000:.0f}k"))

# Add legend (positioned to avoid overlap)
ax.legend(loc="upper left", fontsize=11, frameon=True, fancybox=True, shadow=True)

# Add text box for growth rate (bottom right)
ax.text(
    0.98, 0.02,
    f"Annual Growth: {annual_growth:,} people/year\nR² (Trend Fit): {r_squared}",
    transform=ax.transAxes, ha="right", va="bottom",
    fontsize=11, fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.8)
)

# Adjust layout to prevent label cutoff
plt.tight_layout()

# Save the graph (high resolution for publications)
plt.savefig("Gasabo_Population_2000_2032.png", dpi=300, bbox_inches="tight")
plt.close()

# Print confirmation + key stats
print("Graph saved as 'Gasabo_Population_2000_2032_1.png'")
print(f"\nKey Statistics (2000–2032):")
print(f"- Total Population Growth: {df[df['Year']==2032]['Population'].values[0] - df[df['Year']==2000]['Population'].values[0]:,} people")
print(f"- Annual Growth Rate: {annual_growth:,} people/year")
print(f"- Trendline R²: {r_squared} (higher = stronger linear trend)")

Graph saved as 'Gasabo_Population_2000_2032_1.png'

Key Statistics (2000–2032):
- Total Population Growth: 1,125,240 people
- Annual Growth Rate: 34,467 people/year
- Trendline R²: 0.964 (higher = stronger linear trend)
